## 0. Kernel setup (run in a terminal, not in this notebook)

Before launching this notebook, create and select a conda environment kernel (`2ndWorkshop`).

### Purdue Gilbreth cluster

```bash
module load conda
conda-env-mod create -n ENV_NAME_HERE -j
module use $HOME/privatemodules
module load conda-env/ENV_NAME_HERE-py3.10.11
```

Replace `ENV_NAME_HERE` with your environment name (`2ndWorkshop`), then select the matching kernel in Jupyter before running the cells below.

Check the env and kernel were created:
```bash
conda env list
jupyter kernelspec list
```

### Local machine (conda)

```bash
conda create -n 2ndWorkshop python=3.10 pip -y
conda activate 2ndWorkshop
pip install ipykernel
python -m ipykernel install --user --name 2ndWorkshop --display-name "Python (2ndWorkshop)"
```

Select the `Python (2ndWorkshop)` kernel, run the install cell below once, then restart the kernel.

**Note:** This notebook only starts the MCP tool server — it doesn't need an LLM backend
at all. Keep its kernel running, then open **`Workshop2_Part2b_Agent.ipynb`** in a
*separate* kernel to connect the agent.


# Workshop 2, Part 2a: MCP Tool Server

This is the **server half** of Part 2. It wraps the same course knowledge base and
academic calendar from Part 1 in three tools — search, calendar lookup, and a
notification writer — and exposes them over HTTP using **MCP** (Model Context Protocol),
so any MCP-compatible agent can discover and call them.

**What you'll do:**
- Define the MCP tools with FastMCP
- Start the server — the last cell blocks and keeps it running

Run this notebook first and leave its kernel running. Then open
**`Workshop2_Part2b_Agent.ipynb`** in a separate kernel — that's where the LangGraph
agent connects to these tools and does the reasoning.

Two notebooks, two kernels, two real separate processes — this is how MCP servers are
used in practice (the server could just as easily be running on a different machine).


## 0. Install dependencies

Run once, then restart the kernel.

In [ ]:
! pip install sentence-transformers faiss-cpu fastmcp langchain-text-splitters python-dotenv


---
# Part B: MCP Server — Exposing Tools over HTTP

**MCP (Model Context Protocol)** is a standard for exposing tools that LLMs can call.
This notebook runs the server in the foreground — the last cell (B5) blocks and keeps it
running, the same way a server process keeps running in a terminal.

**Transport options:**

| Transport | How it works | Best for |
|---|---|---|
| `http` (Streamable HTTP) | Agent connects via HTTP; server runs independently | Production, multi-client, debuggable |
| `stdio` | Client spawns server as a subprocess | CLI tools, local single-client use |


## B1. Imports for the MCP server

In [ ]:
import os
import json
from pathlib import Path

import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from fastmcp import FastMCP

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "boilermaker_ta_data"

print("Data directory:", DATA_DIR)
print("Files:", list(DATA_DIR.iterdir()) if DATA_DIR.exists() else "NOT FOUND")


## B2. SimpleRetriever — shared retrieval helper for MCP tools

In [3]:
class SimpleRetriever:
    def __init__(self, texts, metadatas, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.embedder  = SentenceTransformer(model_name)
        self.texts     = texts
        self.metadatas = metadatas
        self.index     = self._build_index(texts)

    def _build_index(self, texts):
        embeddings = self.embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(1, -1)
        faiss.normalize_L2(embeddings)
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)
        return index

    def similarity_search(self, query: str, k: int = 3):
        q_emb = self.embedder.encode(query, convert_to_numpy=True)
        if q_emb.ndim == 1:
            q_emb = q_emb.reshape(1, -1)
        faiss.normalize_L2(q_emb)
        scores, ids = self.index.search(q_emb, min(k, len(self.texts)))
        return [
            type("Doc", (), {"page_content": self.texts[idx], "metadata": self.metadatas[idx]})
            for idx in ids[0]
        ]

## B3. Retriever builder helpers

Both MCP search tools use semantic search (embedding + FAISS) exclusively — no keyword
matching anywhere in this server.

Uses the same chunking approach as Part 1 (`Workshop2_Part1_RAG.ipynb`, section A3) for
the knowledge base: `RecursiveCharacterTextSplitter` (`chunk_size=1200` /
`chunk_overlap=300` characters) splits on paragraph, then sentence, then word boundaries
before falling back to a hard character cut, and is a no-op on text shorter than
`chunk_size` — so the same `chunk_text` call is safe whether a document is one short
paragraph or a long file. Each chunk gets a `Title: ...\nChunk: N\n` header prepended
before embedding, exactly as in Part 1, so a chunk retrieved on its own still carries
its source and position.

Calendar events, by contrast, are already short single-line records, so
`_build_calendar_retriever` flattens each one straight to text with no chunking step.
</cell id="26d67905">


In [ ]:
ANNOUNCEMENTS_FILE = BASE_DIR / "workshop_outputs/announcements.txt"


def _load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def chunk_text(text: str, chunk_size: int = 1200, chunk_overlap: int = 300):
    """Split text into chunks using LangChain's RecursiveCharacterTextSplitter.

    Same chunker as Part 1 (Workshop2_Part1_RAG.ipynb, section A3): tries paragraph,
    then sentence, then word boundaries before falling back to a hard character cut.
    chunk_size / chunk_overlap are in characters. Short text (<= chunk_size chars)
    comes back as a single chunk, so this is safe to call on every document regardless
    of length.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)


def _build_retriever():
    """Build a FAISS retriever from the knowledge base.

    Chunked the same way as Part 1: each document is split with chunk_text, and each
    chunk gets a "Title: ...\\nChunk: N\\n" header prepended before embedding so it
    carries its source and position even when retrieved on its own.
    For production, cache the vectorstore to avoid rebuilding on every call.
    """
    kb = _load_json(DATA_DIR / "knowledge_base.json")
    texts, metadatas = [], []
    for doc in kb:
        title = doc.get("title")
        for chunk_num, chunk in enumerate(chunk_text(doc["text"]), start=1):
            header = f"Title: {title}\nChunk: {chunk_num}\n"
            texts.append(header + chunk)
            metadatas.append({"title": title, "chunk": chunk_num})
    return SimpleRetriever(texts, metadatas)


def _build_calendar_retriever():
    """Build a FAISS retriever from academic calendar events.
    Calendar events are flattened to 'date title notes' strings for embedding. These
    are already short, single-line records, so they aren't run through chunk_text --
    it would be a no-op on them anyway (see Part 1, section A3).
    """
    calendar = _load_json(DATA_DIR / "purdue_calendar.json")
    events   = calendar.get("events", [])
    if not events:
        return None
    texts     = [f"{e['date']} {e['title']} {e['notes']}" for e in events]
    metadatas = [{"date": e.get("date"), "title": e.get("title")} for e in events]
    return SimpleRetriever(texts, metadatas)

## B4. Define the MCP app and its tools

Each `@app.tool` function becomes a tool the LLM agent can call by name.

In [ ]:
app = FastMCP(
    name="boilermaker-ta",
    instructions=(
        "Provides tools for finding course facts, querying the academic calendar, "
        "and writing student notification announcements."
    ),
)

# Built once here, at server startup, instead of inside the tool functions below.
# Rebuilding per-call would reload SentenceTransformer and re-embed everything on every
# single tool invocation -- and since gather_context_node calls search_knowledge_base
# and get_academic_calendar concurrently, two full rebuilds firing at once contend with
# each other badly enough to hang the server. Building once and reusing avoids that.
print("Building knowledge base retriever...")
_KB_RETRIEVER = _build_retriever()
print("Building calendar retriever...")
_CALENDAR_RETRIEVER = _build_calendar_retriever()
print("Retrievers ready.")


@app.tool
def search_knowledge_base(query: str, k: int = 3) -> str:
    """Search the course knowledge base using semantic search (embedding + FAISS)."""
    docs = _KB_RETRIEVER.similarity_search(query, k=k)
    if not docs:
        return "No relevant knowledge found. Try a different question."
    lines = []
    for d in docs:
        title   = d.metadata.get("title") if d.metadata else "(no title)"
        excerpt = (d.page_content[:400] + "...") if len(d.page_content) > 400 else d.page_content
        lines.append(f"{title}: {excerpt}")
    return "\n\n".join(lines)


@app.tool
def get_academic_calendar(query: str = "next 30 days") -> str:
    """Retrieve calendar events using semantic search (embedding + FAISS)."""
    if _CALENDAR_RETRIEVER is None:
        return "Academic calendar is empty."
    docs  = _CALENDAR_RETRIEVER.similarity_search(query, k=5)
    lines = [
        f"{d.metadata.get('date')} - {d.metadata.get('title')}"
        for d in docs
    ]
    return "Academic calendar events:\n" + "\n".join(lines)


@app.tool
def create_notification(subject: str, body: str) -> str:
    """Write a notification to announcements.txt and return its location."""
    ANNOUNCEMENTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    entry = f"Subject: {subject}\n{body}\n---\n"
    with open(ANNOUNCEMENTS_FILE, "a", encoding="utf-8") as f:
        f.write(entry)
    return f"Notification written to {ANNOUNCEMENTS_FILE}."


print("MCP app defined with tools:", ["search_knowledge_base", "get_academic_calendar", "create_notification"])

## B5. Start the MCP server (this cell blocks)

This cell runs the server in the foreground: it keeps executing — blocking this kernel —
until you stop it. That's expected.

**Leave this cell running**, then open `Workshop2_Part2b_Agent.ipynb` in a separate
kernel to connect as a client.

To stop the server: **Kernel → Interrupt** (or restart the kernel).


In [6]:
host = os.environ.get("MCP_HOST", "127.0.0.1")
port = int(os.environ.get("MCP_PORT", "8001"))

print(f"Starting MCP server at http://{host}:{port}/mcp")
print("This cell will keep running -- use Kernel > Interrupt to stop the server.")

# Jupyter's kernel already runs an asyncio event loop, so app.run() (which calls
# anyio.run() to start a new one) raises "Already running asyncio in this thread".
# Use the async variant with top-level await instead.
await app.run_async(transport="http", host=host, port=port)


Starting MCP server at http://127.0.0.1:8001/mcp
This cell will keep running -- use Kernel > Interrupt to stop the server.


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.4.5                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      boilermaker-ta, 3.4.5                       │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[08/04/26 12:10:17] INFO     Starting MCP server 'boilermaker-ta' with transport 'http' on         ]8;id=7025756;file:///opt/miniconda3/envs/2ndWorkshop/lib/python3.10/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=7025757;file:///opt/miniconda3/envs/2ndWorkshop/lib/python3.10/site-packages/fastmcp/server/mixins/transport.py#361\361]8;;\
                             http://127.0.0.1:8001/mcp                                                             

INFO:     Started server process [19413]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:55380 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:55380 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:55504 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55505 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:55506 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55507 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55508 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55509 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55510 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55512 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55511 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:55513 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:55514 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55515 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55516 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55517 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:55518 - "DELETE /mcp HTTP/1.1" 200 OK


...

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9427.87it/s]


INFO:     127.0.0.1:59114 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59115 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59117 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59118 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:59119 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59120 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59121 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59122 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59124 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59125 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:59126 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59127 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59128 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59130 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59131 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:59132 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59133 - "POST /mcp HTTP/1.1" 200 OK


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11637.04it/s]


INFO:     127.0.0.1:59135 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59136 - "DELETE /mcp HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [19413]


---
## Next: Workshop2_Part2b_Agent.ipynb

With this server running, open **`Workshop2_Part2b_Agent.ipynb`** (a separate kernel) to
connect the LangGraph agent and run the Boilermaker TA.
